In [0]:
spark.conf.set(
    "fs.azure.account.key.airlineanalyticsprojsrc.dfs.core.windows.net",
    "storage acc key"
)

In [0]:
from pyspark.sql.functions import avg, count, col, when

# Silver Table read karna aggregation ke liye
silver_flights_df = spark.read.format("delta").load("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_flights")

# Table 1: Airline Delay Analytics
airline_delay_gold = silver_flights_df.groupBy("op_unique_carrier") \
    .agg(avg("arr_delay").alias("avg_arrival_delay"), avg("dep_delay").alias("avg_departure_delay"), count("fl_date").alias("total_flights"))
airline_delay_gold.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/gold/gold_airline_delays")

# Table 2: Airport Delay Probabilities
airport_delay_gold = silver_flights_df.groupBy("origin") \
    .agg(count("fl_date").alias("total_flights"), count(when(col("arr_delay") > 15, 1)).alias("delayed_flights")) \
    .withColumn("delay_probability", (col("delayed_flights") / col("total_flights")) * 100)
airport_delay_gold.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/gold/gold_airport_delays")

# Table 3: Weather Delay Analysis
weather_delay_gold = silver_flights_df.filter(col("weather_delay") > 0).groupBy("month", "origin").agg(avg("weather_delay").alias("avg_weather_delay"))
weather_delay_gold.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/gold/gold_weather_delays")

# Table 4: Aircraft Carrier Delay Analytics
carrier_delay_gold = silver_flights_df.groupBy("op_unique_carrier").agg(avg("carrier_delay").alias("avg_carrier_mechanical_delay"))
carrier_delay_gold.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/gold/gold_carrier_delays")

# Table 5: Daily Flight Cancellation Trends
cancellation_gold = silver_flights_df.groupBy("month", "day_of_month").agg(count(when(col("cancelled") == 1, 1)).alias("total_cancelled_flights"))
cancellation_gold.write.format("delta").mode("overwrite").save("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/gold/gold_cancellations")

print("Gold Layer completely ready with 5 Business/Reporting tables! 🎯")

In [0]:
# 1. Silver Table ko read karenge
silver_flights_df = spark.read.format("delta").load("abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_flights")

# Table 1: Airline Delay Analytics
airline_delay_gold = silver_flights_df.groupBy("op_unique_carrier") \
    .agg(avg("arr_delay").alias("avg_arrival_delay"), avg("dep_delay").alias("avg_departure_delay"), count("fl_date").alias("total_flights"))
airline_delay_gold.write.format("delta").mode("overwrite").saveAsTable("default.gold_airline_delays")

# Table 2: Airport Delay Probabilities
airport_delay_gold = silver_flights_df.groupBy("origin") \
    .agg(count("fl_date").alias("total_flights"), count(when(col("arr_delay") > 15, 1)).alias("delayed_flights")) \
    .withColumn("delay_probability", (col("delayed_flights") / col("total_flights")) * 100)
airport_delay_gold.write.format("delta").mode("overwrite").saveAsTable("default.gold_airport_delays")

# Table 3: Weather Delay Analysis
weather_delay_gold = silver_flights_df.filter(col("weather_delay") > 0).groupBy("month", "origin").agg(avg("weather_delay").alias("avg_weather_delay"))
weather_delay_gold.write.format("delta").mode("overwrite").saveAsTable("default.gold_weather_delays")

# Table 4: Aircraft Carrier Delay Analytics
carrier_delay_gold = silver_flights_df.groupBy("op_unique_carrier").agg(avg("carrier_delay").alias("avg_carrier_mechanical_delay"))
carrier_delay_gold.write.format("delta").mode("overwrite").saveAsTable("default.gold_carrier_delays")

# Table 5: Daily Flight Cancellation Trends
cancellation_gold = silver_flights_df.groupBy("month", "day_of_month").agg(count(when(col("cancelled") == 1, 1)).alias("total_cancelled_flights"))
cancellation_gold.write.format("delta").mode("overwrite").saveAsTable("default.gold_cancellations")

print("Gold Tables successfully registered in Databricks Catalog without external path restrictions! 🎯")

In [0]:
display(spark.sql("DESCRIBE HISTORY delta.`abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_flights`"))